# Cartera eficiente (Markowitz) - Análisis de Carteras

Este notebook hace un análisis financiero completo de un conjunto de acciones del mercado de
capitales argentino (BYMA), pensado para correr en Google Colab sin instalar nada:

1. Lee `empresas_byma.xlsx` directamente desde GitHub y toma las empresas marcadas con **x**.
2. Descarga precios históricos (Yahoo Finance) de esas acciones y de un índice de referencia
   (Merval) para comparar.
3. Calcula estadística descriptiva de retornos (media, volatilidad, asimetría, curtosis, test de
   normalidad), matriz de correlación y clustering jerárquico de activos.
4. Calcula métricas de riesgo por activo: VaR histórico y paramétrico, CVaR (Expected Shortfall),
   Sortino ratio, Calmar ratio, drawdown máximo, Beta y Alpha (CAPM) vs el Merval.
5. Optimiza **seis** carteras con criterios distintos: mínima varianza, máximo Sharpe, máxima
   diversificación, risk parity, HRP (jerárquica) e igual ponderación (1/N) como referencia.
6. Dibuja la **frontera eficiente de Markowitz** (nube de Monte Carlo) con las seis carteras
   marcadas.
7. Compara las seis carteras con una tabla de métricas completa y su contribución al riesgo por
   activo.
8. Corre un **backtest walk-forward** (fuera de muestra, sin look-ahead) de las estrategias de
   mínima varianza y máximo Sharpe re-optimizadas periódicamente, contra 1/N y el Merval.
9. Muestra la **incertidumbre de los pesos óptimos** vía bootstrap (remuestreo de los retornos
   históricos), para visualizar cuánto dependen de la muestra usada.

**Cómo usarlo:** `Entorno de ejecución -> Ejecutar todas` (Runtime -> Run all). No hace falta
tocar nada salvo, opcionalmente, los parámetros de la celda de configuración.

**Importante:** los precios están en pesos argentinos (ARS) nominales, sin ajustar por inflación.
Nada de esto es asesoramiento financiero: es una herramienta de análisis exploratorio que asume
que el pasado (retornos, volatilidad, correlaciones) es una guía razonable del futuro, lo cual no
está garantizado. Ver la sección final de **Notas y limitaciones** antes de sacar conclusiones.

In [ ]:
!pip install -q yfinance openpyxl

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (10, 6)

## Configuración

In [ ]:
# URL raw del Excel en GitHub (rama main). Si cambiás el nombre del archivo o la rama, actualizá esta URL.
EXCEL_URL = "https://raw.githubusercontent.com/alanartola/analisis_carteras/main/empresas_byma.xlsx"

# Ventana histórica usada para estimar retornos y riesgo
LOOKBACK_PERIOD = "3y"   # ej: "1y", "2y", "3y", "5y"

# Tasa libre de riesgo anual (en la misma base que los retornos, ARS nominal). Ajustala si querés.
RISK_FREE_RATE = 0.0

# Índice de referencia (benchmark) para Beta/Alpha (CAPM) y para comparar el backtest. Merval en Yahoo Finance.
BENCHMARK_TICKER = "^MERV"
BENCHMARK_NOMBRE = "Merval"

# Cantidad de carteras aleatorias para dibujar la nube de la frontera eficiente
N_PORTFOLIOS = 20000

# Sin ventas en corto: pesos entre 0% y 100% por activo
ALLOW_SHORT = False

# Límite máximo opcional de concentración por activo (ej. 0.30 = ningún activo puede pesar más del
# 30% de la cartera). None = sin límite adicional (más allá del rango que ya impone ALLOW_SHORT).
MAX_WEIGHT_PER_ASSET = None

# Niveles de confianza para VaR / CVaR (Value at Risk / Expected Shortfall)
VAR_CONFIDENCE_LEVELS = [0.95, 0.99]

# Backtesting walk-forward: ventana de entrenamiento (días hábiles) y frecuencia de rebalanceo
BACKTEST_TRAIN_WINDOW = 252    # ~1 año de historia para estimar retorno/riesgo en cada rebalanceo
BACKTEST_REBALANCE_EVERY = 21  # ~1 mes hábil entre rebalanceos

# Robustez: cantidad de remuestreos bootstrap para estimar la incertidumbre de los pesos óptimos
N_BOOTSTRAP = 300

DIAS_HABILES = 252

## 1. Cargar el Excel y filtrar las empresas marcadas con x

In [ ]:
universo = pd.read_excel(EXCEL_URL, sheet_name="Universo BYMA")

marca = universo["Analizar (x)"].astype(str).str.strip().str.lower()
seleccion = universo[marca.isin(["x", "si", "sí"])].copy()

if seleccion.empty:
    raise ValueError(
        "No hay ninguna empresa marcada con 'x' en la columna 'Analizar (x)' del Excel. "
        "Marcá al menos 2 empresas, subí el cambio a GitHub y volvé a ejecutar el notebook."
    )

print(f"Empresas seleccionadas: {len(seleccion)}")
seleccion[["Ticker", "Empresa", "Sector", "Ticker Yahoo Finance"]]

## 2. Descargar precios históricos (acciones + índice de referencia)

In [ ]:
tickers = seleccion["Ticker Yahoo Finance"].tolist()
nombres = dict(zip(seleccion["Ticker Yahoo Finance"], seleccion["Ticker"]))

raw = yf.download(tickers, period=LOOKBACK_PERIOD, auto_adjust=True, progress=False)
precios = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]].rename(columns={"Close": tickers[0]})

# Descarta columnas sin datos suficientes (ticker sin cotización, delisted, etc.)
precios = precios.dropna(axis=1, thresh=int(len(precios) * 0.7))
faltantes = set(tickers) - set(precios.columns)
if faltantes:
    print(f"Aviso: sin datos suficientes para {sorted(faltantes)}; se excluyen del análisis.")

precios = precios.rename(columns=nombres).dropna().sort_index()

if precios.shape[1] < 2:
    raise ValueError("Se necesitan al menos 2 acciones con datos válidos para calcular una cartera. Marcá más empresas en el Excel.")

# Índice de referencia (benchmark), usado más adelante para Beta/Alpha (CAPM) y el backtest.
# Si falla la descarga, el resto del notebook sigue funcionando sin él (queda documentado el aviso).
try:
    bench_raw = yf.download(BENCHMARK_TICKER, period=LOOKBACK_PERIOD, auto_adjust=True, progress=False)
    bench_precios = bench_raw["Close"]
    if isinstance(bench_precios, pd.DataFrame):
        bench_precios = bench_precios.iloc[:, 0]
    bench_precios = bench_precios.dropna().sort_index()
    tiene_benchmark = len(bench_precios) > 0
except Exception as e:
    print(f"Aviso: no se pudo descargar el benchmark {BENCHMARK_TICKER} ({e}). Se continúa sin Beta/Alpha/comparación contra el índice.")
    bench_precios = None
    tiene_benchmark = False

precios.tail()

## 3. Retornos, riesgo y estadística descriptiva (anualizados)

Además de la media y el desvío estándar (los dos únicos insumos que usa Markowitz), se calcula la
**asimetría** y la **curtosis en exceso** de los retornos diarios, y el test de normalidad
**Jarque-Bera**. Un p-valor bajo (< 0.05) rechaza la hipótesis de normalidad: es habitual en
acciones individuales y es la razón por la que, más adelante, se complementa el análisis de
varianza con VaR, CVaR y drawdown máximo (medidas que no asumen una distribución normal).

In [ ]:
retornos_diarios = precios.pct_change().dropna()

retornos_anuales = retornos_diarios.mean() * DIAS_HABILES
cov_anual = retornos_diarios.cov() * DIAS_HABILES
vol_anual = pd.Series(np.sqrt(np.diag(cov_anual)), index=cov_anual.index)


def max_drawdown(serie_precios):
    acumulado = serie_precios / serie_precios.iloc[0]
    pico = acumulado.cummax()
    return (acumulado / pico - 1.0).min()


max_dd_por_activo = precios.apply(max_drawdown)
asimetria = retornos_diarios.skew()
curtosis_exceso = retornos_diarios.kurt()

jb_pvalor = {}
for col in retornos_diarios.columns:
    _, p = stats.jarque_bera(retornos_diarios[col].values)
    jb_pvalor[col] = p

resumen = pd.DataFrame({
    "Retorno anual esperado": retornos_anuales,
    "Volatilidad anual": vol_anual,
    "Asimetría (retornos diarios)": asimetria,
    "Curtosis exceso (retornos diarios)": curtosis_exceso,
    "Jarque-Bera p-valor": pd.Series(jb_pvalor),
    "Drawdown máximo histórico": max_dd_por_activo,
})
resumen.sort_values("Retorno anual esperado", ascending=False)

In [ ]:
# Alinear el benchmark a las mismas fechas que los precios (relleno hacia adelante en feriados
# que no coinciden entre el índice y alguna acción), para poder comparar manzanas con manzanas.
if tiene_benchmark:
    bench_precios_alineado = bench_precios.reindex(precios.index).ffill()
    bench_ret = bench_precios_alineado.pct_change().reindex(retornos_diarios.index)
else:
    bench_ret = None

## 4. Matriz de correlación y clustering jerárquico

La correlación entre activos es lo que determina cuánto beneficio real de diversificación aporta
combinarlos: dos acciones con retorno y volatilidad parecidos pero baja correlación reducen más el
riesgo de la cartera que dos acciones muy correlacionadas. El **dendrograma** agrupa los activos
que históricamente se movieron parecido (se unen a menor distancia): activos que se juntan muy
temprano en el árbol aportan poca diversificación entre sí.

In [ ]:
corr = retornos_diarios.corr()

fig, ax = plt.subplots(figsize=(1.0 * len(corr.columns) + 3, 1.0 * len(corr.columns) + 2))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        valor = corr.values[i, j]
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(valor) > 0.6 else "black")
fig.colorbar(im, label="Correlación", shrink=0.8)
ax.set_title("Matriz de correlación de retornos diarios")
plt.tight_layout()
plt.show()

In [ ]:
distancia_corr = np.sqrt(np.clip((1 - corr.values) / 2, 0, None))
np.fill_diagonal(distancia_corr, 0.0)
condensada = squareform(distancia_corr, checks=False)
enlace = linkage(condensada, method="average")

plt.figure(figsize=(max(10, 0.5 * len(corr.columns)), 6))
dendrogram(enlace, labels=corr.columns.tolist(), leaf_rotation=90)
plt.title("Clustering jerárquico de activos (distancia basada en correlación)")
plt.ylabel("Distancia")
plt.tight_layout()
plt.show()

## 5. Métricas de riesgo por activo

- **VaR histórico** (Value at Risk): pérdida diaria que, históricamente, solo se superó en el
  `(1 - confianza)`% peor de los días. Se calcula directamente del percentil de los retornos
  observados, sin asumir ninguna distribución.
- **CVaR / Expected Shortfall**: pérdida diaria *promedio* en esos peores días (más informativo
  que el VaR porque no ignora qué tan mala es la cola).
- **VaR paramétrico**: la misma idea pero asumiendo retornos normales (media y desvío diarios) —
  se incluye solo a modo de comparación; dado el test de Jarque-Bera de la sección anterior, suele
  subestimar el riesgo de cola real.
- **Sortino ratio**: como el Sharpe ratio, pero penaliza solo la volatilidad *a la baja*
  (`downside deviation`), no la volatilidad al alza (que ningún inversor considera "riesgo").
- **Calmar ratio**: retorno anual dividido por el drawdown máximo histórico (en valor absoluto).
- **Beta / Alpha (CAPM)**: sensibilidad del activo a los movimientos del Merval (Beta) y el
  retorno anualizado que no se explica por esa sensibilidad (Alpha), estimados por regresión
  lineal simple sobre los retornos diarios.

In [ ]:
def var_historico(retornos, confianza):
    return -np.percentile(retornos, (1 - confianza) * 100)


def cvar_historico(retornos, confianza):
    umbral = np.percentile(retornos, (1 - confianza) * 100)
    cola = retornos[retornos <= umbral]
    return -cola.mean() if len(cola) > 0 else np.nan


def var_parametrico(media, desvio, confianza):
    z = stats.norm.ppf(1 - confianza)
    return -(media + z * desvio)


def downside_deviation(retornos, objetivo_minimo=0.0):
    bajo_objetivo = retornos[retornos < objetivo_minimo] - objetivo_minimo
    if len(bajo_objetivo) == 0:
        return 0.0
    return float(np.sqrt((bajo_objetivo ** 2).mean()))


filas_riesgo = []
for col in retornos_diarios.columns:
    r = retornos_diarios[col].values
    fila = {"Activo": col}
    for cl in VAR_CONFIDENCE_LEVELS:
        fila[f"VaR histórico diario {int(cl * 100)}%"] = var_historico(r, cl)
        fila[f"CVaR histórico diario {int(cl * 100)}%"] = cvar_historico(r, cl)
    fila["VaR paramétrico diario 95%"] = var_parametrico(r.mean(), r.std(), 0.95)

    dd_anual = downside_deviation(r) * np.sqrt(DIAS_HABILES)
    ret_anual = retornos_anuales[col]
    fila["Downside deviation anual"] = dd_anual
    fila["Sortino ratio"] = (ret_anual - RISK_FREE_RATE) / dd_anual if dd_anual > 0 else np.nan

    mdd = max_dd_por_activo[col]
    fila["Calmar ratio"] = ret_anual / abs(mdd) if mdd < 0 else np.nan
    filas_riesgo.append(fila)

tabla_riesgo = pd.DataFrame(filas_riesgo).set_index("Activo")

if tiene_benchmark:
    conjunto_capm = retornos_diarios.join(bench_ret.rename(BENCHMARK_NOMBRE), how="inner")
    betas, alphas = {}, {}
    for col in retornos_diarios.columns:
        datos = conjunto_capm[[col, BENCHMARK_NOMBRE]].dropna()
        beta, alpha_diario = np.polyfit(datos[BENCHMARK_NOMBRE].values, datos[col].values, 1)
        betas[col] = beta
        alphas[col] = alpha_diario * DIAS_HABILES
    tabla_riesgo[f"Beta vs {BENCHMARK_NOMBRE}"] = pd.Series(betas)
    tabla_riesgo[f"Alpha anualizado vs {BENCHMARK_NOMBRE}"] = pd.Series(alphas)
else:
    print("Sin datos de benchmark: se omiten Beta/Alpha (CAPM) por activo.")

tabla_riesgo

## 6. Optimización de carteras: seis criterios distintos

In [ ]:
activos = list(precios.columns)
n = len(activos)
mu = retornos_anuales.values
cov = cov_anual.values
sigma_individual = vol_anual.values


def rendimiento_cartera(w):
    return float(np.dot(w, mu))


def volatilidad_cartera(w):
    return float(np.sqrt(np.dot(w, np.dot(cov, w))))


def sharpe_negativo(w):
    vol = volatilidad_cartera(w)
    return 0.0 if vol == 0 else -(rendimiento_cartera(w) - RISK_FREE_RATE) / vol


def diversification_ratio(w):
    vol = volatilidad_cartera(w)
    if vol == 0:
        return 0.0
    return float(np.dot(w, sigma_individual)) / vol


def diversification_ratio_negativo(w):
    return -diversification_ratio(w)


def contribuciones_riesgo(w):
    # Contribución absoluta de cada activo a la volatilidad total de la cartera (suman volatilidad_cartera(w)).
    vol = volatilidad_cartera(w)
    if vol == 0:
        return np.zeros_like(w)
    marginal = np.dot(cov, w) / vol
    return w * marginal


def objetivo_risk_parity(w):
    vol = volatilidad_cartera(w)
    if vol == 0:
        return 1e6
    contrib_pct = contribuciones_riesgo(w) / vol
    return float(np.sum((contrib_pct - contrib_pct.mean()) ** 2))


cota_superior = 1.0 if MAX_WEIGHT_PER_ASSET is None else min(1.0, MAX_WEIGHT_PER_ASSET)
cota_inferior = -1.0 if ALLOW_SHORT else 0.0
bounds = tuple((cota_inferior, cota_superior) for _ in range(n))
# Risk parity y máxima diversificación asumen contribuciones de riesgo positivas: no admiten ventas en corto.
bounds_largo_solo = tuple((0.0, cota_superior) for _ in range(n))
restricciones = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
w0 = np.repeat(1.0 / n, n)

opt_min_var = minimize(volatilidad_cartera, w0, method="SLSQP", bounds=bounds, constraints=restricciones)
opt_max_sharpe = minimize(sharpe_negativo, w0, method="SLSQP", bounds=bounds, constraints=restricciones)
opt_max_div = minimize(diversification_ratio_negativo, w0, method="SLSQP", bounds=bounds_largo_solo, constraints=restricciones)
opt_risk_parity = minimize(objetivo_risk_parity, w0, method="SLSQP", bounds=bounds_largo_solo, constraints=restricciones)

pesos_min_var = opt_min_var.x
pesos_max_sharpe = opt_max_sharpe.x
pesos_max_div = opt_max_div.x
pesos_risk_parity = opt_risk_parity.x / opt_risk_parity.x.sum()

print("Mínima varianza:       ", "OK" if opt_min_var.success else opt_min_var.message)
print("Máximo Sharpe:         ", "OK" if opt_max_sharpe.success else opt_max_sharpe.message)
print("Máxima diversificación:", "OK" if opt_max_div.success else opt_max_div.message)
print("Risk parity:           ", "OK" if opt_risk_parity.success else opt_risk_parity.message)

### Hierarchical Risk Parity (HRP)

El método de Markowitz clásico invierte la matriz de covarianza, lo que lo hace muy sensible a
errores de estimación cuando hay muchos activos correlacionados entre sí (Michaud, 1989 - "error
maximization"). **HRP** (López de Prado, 2016) evita ese problema: agrupa los activos por
similitud (el mismo clustering jerárquico de la sección anterior) y reparte el capital de forma
recursiva entre los clusters según su varianza, sin invertir nunca la matriz de covarianza
completa. Suele dar carteras más estables y mejor diversificadas fuera de muestra que Markowitz
clásico, a costa de no ser estrictamente "óptima" dentro de muestra.

In [ ]:
def _orden_quasi_diagonal(enlace_link):
    enlace_link = enlace_link.astype(int)
    n_items = enlace_link[-1, 3]
    orden = pd.Series([enlace_link[-1, 0], enlace_link[-1, 1]])
    while orden.max() >= n_items:
        orden.index = range(0, orden.shape[0] * 2, 2)
        es_cluster = orden[orden >= n_items]
        idx = es_cluster.index
        fila_enlace = es_cluster.values - n_items
        orden[idx] = enlace_link[fila_enlace, 0]
        hijos = pd.Series(enlace_link[fila_enlace, 1], index=idx + 1)
        orden = pd.concat([orden, hijos]).sort_index()
        orden.index = range(orden.shape[0])
    return orden.tolist()


def _varianza_cluster(cov_df, items):
    cov_sub = cov_df.loc[items, items].values
    ivp = 1.0 / np.diag(cov_sub)
    ivp = ivp / ivp.sum()
    return float(ivp @ cov_sub @ ivp)


def _biparticion_recursiva(cov_df, items_ordenados):
    pesos = pd.Series(1.0, index=items_ordenados)
    clusters = [items_ordenados]
    while len(clusters) > 0:
        clusters = [c[i:j] for c in clusters
                    for i, j in ((0, len(c) // 2), (len(c) // 2, len(c)))
                    if len(c) > 1]
        for i in range(0, len(clusters), 2):
            c0, c1 = clusters[i], clusters[i + 1]
            var0 = _varianza_cluster(cov_df, c0)
            var1 = _varianza_cluster(cov_df, c1)
            alfa = 1 - var0 / (var0 + var1)
            pesos[c0] *= alfa
            pesos[c1] *= (1 - alfa)
    return pesos


def pesos_hrp(retornos):
    corr_hrp = retornos.corr()
    cov_hrp = retornos.cov()
    dist = np.sqrt(np.clip((1 - corr_hrp.values) / 2, 0, None))
    np.fill_diagonal(dist, 0.0)
    enlace_hrp = linkage(squareform(dist, checks=False), method="single")
    orden = _orden_quasi_diagonal(enlace_hrp)
    etiquetas = [retornos.columns[i] for i in orden]
    serie_pesos = _biparticion_recursiva(cov_hrp, etiquetas)
    return serie_pesos.reindex(retornos.columns).values


pesos_hrp_valores = pesos_hrp(retornos_diarios)
print("HRP - suma de pesos (debe ser ~1):", round(pesos_hrp_valores.sum(), 6))

## 7. Frontera eficiente (nube de carteras simuladas)

In [ ]:
rng = np.random.default_rng(42)
resultados = np.zeros((N_PORTFOLIOS, 3))

for i in range(N_PORTFOLIOS):
    w = rng.normal(size=n) if ALLOW_SHORT else rng.random(n)
    w = w / np.sum(w)
    ret = rendimiento_cartera(w)
    vol = volatilidad_cartera(w)
    sharpe = (ret - RISK_FREE_RATE) / vol if vol > 0 else 0.0
    resultados[i] = [vol, ret, sharpe]

carteras_marcadas = {
    "Mínima varianza": (pesos_min_var, "red", "*"),
    "Máximo Sharpe": (pesos_max_sharpe, "gold", "*"),
    "Máx. diversificación": (pesos_max_div, "deepskyblue", "D"),
    "Risk parity": (pesos_risk_parity, "limegreen", "D"),
    "HRP": (pesos_hrp_valores, "darkorchid", "D"),
    "Igual ponderación (1/N)": (w0, "black", "o"),
}

plt.figure(figsize=(11, 7))
sc = plt.scatter(resultados[:, 0], resultados[:, 1], c=resultados[:, 2], cmap="viridis", s=6, alpha=0.4)
plt.colorbar(sc, label="Sharpe ratio")

for nombre, (w, color, marcador) in carteras_marcadas.items():
    plt.scatter(volatilidad_cartera(w), rendimiento_cartera(w), marker=marcador, color=color,
                edgecolor="black", s=250, label=nombre, zorder=5)

plt.xlabel("Volatilidad anual (desvío estándar)")
plt.ylabel("Retorno anual esperado")
plt.title("Frontera eficiente de Markowitz")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Comparación de las seis carteras: composición y métricas

In [ ]:
def metricas_cartera(w):
    ret = rendimiento_cartera(w)
    vol = volatilidad_cartera(w)
    sharpe = (ret - RISK_FREE_RATE) / vol if vol > 0 else np.nan

    ret_diarios_cartera = retornos_diarios.values @ w
    serie_ret_cartera = pd.Series(ret_diarios_cartera, index=retornos_diarios.index)

    dd_anual = downside_deviation(ret_diarios_cartera) * np.sqrt(DIAS_HABILES)
    sortino = (ret - RISK_FREE_RATE) / dd_anual if dd_anual > 0 else np.nan

    valor_cartera = (1 + serie_ret_cartera).cumprod()
    mdd = (valor_cartera / valor_cartera.cummax() - 1.0).min()
    calmar = ret / abs(mdd) if mdd < 0 else np.nan

    fila = {
        "Retorno anual esperado": ret,
        "Volatilidad anual": vol,
        "Sharpe ratio": sharpe,
        "Sortino ratio": sortino,
        "Calmar ratio": calmar,
        "VaR histórico diario 95%": var_historico(ret_diarios_cartera, 0.95),
        "CVaR histórico diario 95%": cvar_historico(ret_diarios_cartera, 0.95),
        "Máximo drawdown histórico": mdd,
        "Diversification ratio": diversification_ratio(w),
    }
    if tiene_benchmark:
        conjunto = pd.concat([serie_ret_cartera.rename("cartera"), bench_ret.rename(BENCHMARK_NOMBRE)], axis=1).dropna()
        beta_c, alpha_c = np.polyfit(conjunto[BENCHMARK_NOMBRE].values, conjunto["cartera"].values, 1)
        fila[f"Beta vs {BENCHMARK_NOMBRE}"] = beta_c
        fila[f"Alpha anualizado vs {BENCHMARK_NOMBRE}"] = alpha_c * DIAS_HABILES
    return fila


carteras = {
    "Mínima varianza": pesos_min_var,
    "Máximo Sharpe": pesos_max_sharpe,
    "Máx. diversificación": pesos_max_div,
    "Risk parity": pesos_risk_parity,
    "HRP": pesos_hrp_valores,
    "Igual ponderación (1/N)": w0,
}

tabla_pesos_pct = (pd.DataFrame(carteras, index=activos) * 100).round(2)
print("Composición de cada cartera (% del capital):")
display(tabla_pesos_pct)

tabla_metricas = pd.DataFrame({nombre: metricas_cartera(w) for nombre, w in carteras.items()}).T
print("\nMétricas de cada cartera:")
display(tabla_metricas)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (nombre, w) in zip(axes.flat, carteras.items()):
    pesos_pct = tabla_pesos_pct[nombre]
    pesos_no_cero = pesos_pct[pesos_pct > 0.5]
    ax.pie(pesos_no_cero, labels=pesos_no_cero.index, autopct="%1.1f%%", startangle=90, textprops={"fontsize": 8})
    ax.set_title(nombre)
plt.suptitle("Composición de cada cartera")
plt.tight_layout()
plt.show()

## 9. Contribución al riesgo por activo

No es lo mismo el peso de un activo en la cartera que cuánto *riesgo* aporta: un activo con poco
peso pero muy volátil (o muy correlacionado con el resto) puede explicar una porción
desproporcionada de la volatilidad total. La tabla siguiente muestra qué porcentaje de la
volatilidad de cada cartera explica cada activo (las columnas suman 100%).

In [ ]:
tabla_contribucion = pd.DataFrame({
    nombre: (contribuciones_riesgo(w) / volatilidad_cartera(w) * 100) if volatilidad_cartera(w) > 0
    else np.zeros(n)
    for nombre, w in carteras.items()
}, index=activos).round(2)
tabla_contribucion

## 10. Backtest walk-forward (fuera de muestra)

A diferencia de las secciones anteriores (que optimizan una sola vez con todo el historial), acá
se simula cómo le habría ido a un inversor que, cada `BACKTEST_REBALANCE_EVERY` días hábiles,
re-optimiza la cartera usando **solo los datos de los `BACKTEST_TRAIN_WINDOW` días previos** (nunca
información futura) y mantiene esos pesos hasta el próximo rebalanceo. Se compara contra 1/N y
contra el Merval. Esto es lo más parecido a una prueba real de las estrategias: la optimización
"perfecta dentro de muestra" de las secciones 6-9 típicamente pierde performance fuera de muestra
(sobreajuste a la historia usada para estimarla) — por eso importa mirar ambas cosas.

In [ ]:
def _optimizar_ventana(mu_ventana, cov_ventana, objetivo):
    n_local = len(mu_ventana)
    bounds_local = tuple((cota_inferior, cota_superior) for _ in range(n_local))
    restricciones_local = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    w0_local = np.repeat(1.0 / n_local, n_local)

    def vol_local(w):
        return float(np.sqrt(np.dot(w, np.dot(cov_ventana, w))))

    def objetivo_local(w):
        if objetivo == "min_var":
            return vol_local(w)
        v = vol_local(w)
        return 0.0 if v == 0 else -(float(np.dot(w, mu_ventana)) - RISK_FREE_RATE) / v

    res = minimize(objetivo_local, w0_local, method="SLSQP", bounds=bounds_local, constraints=restricciones_local)
    return res.x if res.success else w0_local


retornos_bt = retornos_diarios  # ya está ordenado por fecha y sin NaN
fechas_bt_totales = retornos_bt.index
inicio = BACKTEST_TRAIN_WINDOW

if inicio >= len(fechas_bt_totales):
    print(f"Aviso: se necesitan más de {BACKTEST_TRAIN_WINDOW} días hábiles de historia para el "
          "backtest walk-forward (ampliá LOOKBACK_PERIOD). Se omite esta sección.")
    backtest_disponible = False
else:
    backtest_disponible = True
    estrategias = ["min_var", "max_sharpe"]
    pesos_actuales = {estr: np.repeat(1.0 / n, n) for estr in estrategias}
    filas_bt = {estr: [] for estr in estrategias}
    filas_bt["Igual ponderación (1/N)"] = []
    fechas_out = []

    for t in range(inicio, len(fechas_bt_totales)):
        if (t - inicio) % BACKTEST_REBALANCE_EVERY == 0:
            ventana = retornos_bt.iloc[t - BACKTEST_TRAIN_WINDOW:t]
            mu_ventana = ventana.mean().values * DIAS_HABILES
            cov_ventana = ventana.cov().values * DIAS_HABILES
            for estr in estrategias:
                pesos_actuales[estr] = _optimizar_ventana(mu_ventana, cov_ventana, estr)

        r_dia = retornos_bt.iloc[t].values
        for estr in estrategias:
            filas_bt[estr].append(float(np.dot(pesos_actuales[estr], r_dia)))
        filas_bt["Igual ponderación (1/N)"].append(float(np.mean(r_dia)))
        fechas_out.append(fechas_bt_totales[t])

    nombres_bt = {"min_var": "Mínima varianza (walk-forward)", "max_sharpe": "Máximo Sharpe (walk-forward)"}
    bt_retornos = pd.DataFrame({nombres_bt.get(k, k): v for k, v in filas_bt.items()}, index=fechas_out)

    if tiene_benchmark:
        bt_retornos[BENCHMARK_NOMBRE] = bench_ret.reindex(bt_retornos.index)

    bt_valor = (1 + bt_retornos.fillna(0.0)).cumprod()

    plt.figure(figsize=(12, 6))
    for col in bt_valor.columns:
        plt.plot(bt_valor.index, bt_valor[col], label=col)
    plt.title("Backtest walk-forward: valor de $1 invertido (fuera de muestra, sin costos de transacción)")
    plt.ylabel("Valor acumulado")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    dias_bt = len(bt_retornos)
    resumen_bt = pd.DataFrame({
        "Retorno total": bt_valor.iloc[-1] - 1,
        "Retorno anualizado (CAGR)": bt_valor.iloc[-1] ** (DIAS_HABILES / dias_bt) - 1,
        "Volatilidad anualizada": bt_retornos.std() * np.sqrt(DIAS_HABILES),
        "Sharpe (fuera de muestra)": (bt_retornos.mean() * DIAS_HABILES - RISK_FREE_RATE) / (bt_retornos.std() * np.sqrt(DIAS_HABILES)),
        "Máximo drawdown": (bt_valor / bt_valor.cummax() - 1).min(),
    })
    print(f"\nPeríodo fuera de muestra: {fechas_out[0].date()} a {fechas_out[-1].date()} ({dias_bt} ruedas, {(len(fechas_bt_totales) - inicio) // BACKTEST_REBALANCE_EVERY} rebalanceos)")
    display(resumen_bt)

## 11. Robustez: incertidumbre de los pesos óptimos (bootstrap)

La cartera de máximo Sharpe de la sección 6 es la *mejor posible* asumiendo que `mu` (retornos
esperados) y `cov` (matriz de covarianza) estimados son los correctos — pero son estimaciones
sobre una muestra histórica finita, con su propio error. Acá se remuestrea (bootstrap, con
reemplazo) la serie de retornos históricos `N_BOOTSTRAP` veces, se recalculan `mu`/`cov` y se
re-optimiza la cartera de máximo Sharpe en cada remuestreo, para visualizar cuánto varían los
pesos óptimos solo por el ruido de la muestra (el problema de "maximización del error" de Michaud,
1989 — motivo por el cual HRP y risk parity suelen preferirse quirúrgicamente en la práctica).

In [ ]:
rng_bootstrap = np.random.default_rng(123)
n_obs = len(retornos_diarios)
retornos_matriz = retornos_diarios.values
pesos_bootstrap = np.zeros((N_BOOTSTRAP, n))

for b in range(N_BOOTSTRAP):
    idx = rng_bootstrap.integers(0, n_obs, size=n_obs)
    muestra = retornos_matriz[idx]
    mu_b = muestra.mean(axis=0) * DIAS_HABILES
    cov_b = np.cov(muestra, rowvar=False) * DIAS_HABILES
    pesos_bootstrap[b] = _optimizar_ventana(mu_b, cov_b, "max_sharpe")

pesos_bootstrap_pct = pd.DataFrame(pesos_bootstrap * 100, columns=activos)

plt.figure(figsize=(max(10, 0.5 * n), 6))
plt.boxplot([pesos_bootstrap_pct[c] for c in activos], tick_labels=activos, showfliers=False)
plt.xticks(rotation=90)
plt.ylabel("Peso óptimo (%) en la cartera de máximo Sharpe")
plt.title(f"Incertidumbre de los pesos óptimos ({N_BOOTSTRAP} remuestreos bootstrap)")
plt.axhline(0, color="gray", linewidth=0.5)
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("Interpretación: una caja/bigotes angostos indica un peso óptimo estable ante pequeños cambios")
print("en la muestra histórica. Una caja muy ancha (o que cruza el 0%) indica un peso poco confiable:")
print("pequeñas diferencias en el período usado para estimar mu/cov podrían llevarlo a otro extremo.")

## Notas y limitaciones

**Sobre los datos:**
- Los retornos se calculan sobre precios **nominales en pesos argentinos**, sin ajustar por
  inflación ni por dividendos más allá de lo que `yfinance` ajusta automáticamente
  (`auto_adjust=True`).
- Las 82 empresas de BYMA no cotizan todas con la misma liquidez: acciones poco líquidas pueden
  tener precios "stale" (sin cambios varios días) que subestiman artificialmente su volatilidad y
  su correlación con el resto — revisar el volumen operado antes de confiar en sus métricas.

**Sobre los supuestos del modelo:**
- Markowitz (mínima varianza, máximo Sharpe) asume que el futuro se parece al pasado (usa media y
  covarianza históricas) y que los retornos siguen aproximadamente una distribución normal. La
  sección 3 (test de Jarque-Bera) típicamente muestra que esto no es así para acciones
  individuales — de ahí que se complementen con VaR/CVaR histórico (no paramétrico), Sortino y
  drawdown máximo.
- HRP y risk parity son alternativas menos sensibles al error de estimación de `mu`/`cov`, pero
  **no maximizan explícitamente el Sharpe ratio esperado**: son un compromiso distinto entre
  robustez y "optimalidad" teórica. No hay un método que sea universalmente mejor; por eso se
  muestran los seis lado a lado.
- `RISK_FREE_RATE` es un supuesto que vos definís (0% por defecto); ajustalo a una tasa libre de
  riesgo razonable en pesos (ej. una LECAP corta) para que el Sharpe/Sortino ratio sea más
  representativo.
- Beta/Alpha (CAPM) se calculan por regresión lineal simple sobre datos históricos: el Alpha
  histórico **no** es una predicción de que ese exceso de retorno se vaya a repetir.

**Sobre el backtest walk-forward:**
- No modela costos de transacción, impuestos, ni slippage: cada rebalanceo es gratis e
  instantáneo en la simulación, lo cual sobreestima el retorno real de una estrategia que rebalancea
  seguido.
- El bootstrap de la sección 11 remuestrea días de forma independiente (i.i.d.), lo cual ignora la
  autocorrelación y el agrupamiento de volatilidad (`volatility clustering`) que sí existen en la
  realidad; es una simplificación estándar pero vale aclararla.
- No hay sesgo de supervivencia relevante en este universo (todas las 80 empresas de BYMA
  actualmente activas están incluidas en `empresas_byma.xlsx`), pero si algún ticker se deslistara
  en el futuro, el backtest de ese período seguiría sin poder reconstruir su cotización pasada.

**Sobre las restricciones de la optimización:**
- Sin ventas en corto por defecto (`ALLOW_SHORT = False`); pesos entre 0% y 100% por activo.
- `MAX_WEIGHT_PER_ASSET` permite poner un techo de concentración por activo (ej. `0.30`), pero solo
  afecta a las carteras que se resuelven como optimización con restricciones (mínima varianza,
  máximo Sharpe, máxima diversificación, risk parity). **HRP e 1/N no lo respetan**: HRP reparte el
  capital por bisección recursiva (no por una optimización con límites) y podría, en teoría, superar
  ese techo en algún activo; 1/N reparte siempre en partes iguales.
- No hay (todavía) límites de concentración por sector — se podría agregar restringiendo la suma de
  pesos de cada grupo de `Sector` en las carteras optimizadas, si hiciera falta.

**En general:** esto es una herramienta de análisis exploratorio, no una recomendación de
inversión ni asesoramiento financiero.